In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

# This will automatically pull the credentials you just saved in ~/.qiskit
service = QiskitRuntimeService()

print(f"✅ Successfully authenticated!")
print("Available backends:")
for backend in service.backends(simulator=False, operational=True):
    print(f" - {backend.name} ({backend.num_qubits} qubits)")

qiskit_runtime_service.__init__:WARNING:2026-06-08 16:12:49,782: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-08 16:12:49,782: Loading instance: open-instance, plan: open


✅ Successfully authenticated!
Available backends:
 - ibm_marrakesh (156 qubits)
 - ibm_kingston (156 qubits)
 - ibm_fez (156 qubits)


In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

# 1. Connect using your saved credentials
service = QiskitRuntimeService()

print("🔍 Searching for the least busy quantum computer...")

# 2. Find the fastest available physical hardware that fits our circuit
backend = service.least_busy(
    operational=True, 
    simulator=False, 
    min_num_qubits=9  # We need 8 data qubits + 1 ancilla qubit
)

print(f"✅ Hardware selected!")
print(f"🖥️ Name: {backend.name}")
print(f"📊 Qubits: {backend.num_qubits}")
print(f"⏳ Pending Jobs: {backend.status().pending_jobs}")

qiskit_runtime_service.__init__:WARNING:2026-06-08 16:14:51,819: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


🔍 Searching for the least busy quantum computer...


qiskit_runtime_service.backends:WARNING:2026-06-08 16:14:52,202: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-06-08 16:14:53,277: Using instance: open-instance, plan: open


✅ Hardware selected!
🖥️ Name: ibm_marrakesh
📊 Qubits: 156
⏳ Pending Jobs: 0


## Understanding the Modern Qiskit Execution Path

If you look at older quantum computing textbooks or tutorials (pre-2024), you will often see circuits executed using `execute(circuit, backend)` or `backend.run(circuit)`. However, modern Qiskit has deprecated this approach in favor of **Qiskit Runtime Primitives** (specifically `SamplerV2` and `EstimatorV2`).

Before we can pass our Deutsch-Jozsa circuit to the `SamplerV2`, we must **transpile** it using a Pass Manager (`generate_preset_pass_manager`). Unlike running on a local simulator, transpilation is absolutely mandatory when executing on physical hardware for two critical reasons:

1. **Native Gate Sets:** We wrote our circuit using abstract logical gates like `H` (Hadamard), `X` (Pauli-X), and `CX` (CNOT). However, physical superconducting chips do not natively speak these gates. They use microwave pulses calibrated to specific "Basis Gates" (often `RZ`, `SX`, `X`, and `ECR` or `CZ`). The transpiler acts as a compiler, decomposing our logical gates into the hardware's native pulse instructions.
2. **Qubit Connectivity (Topology):** In our idealized circuit, we assumed any qubit could interact with any other qubit. Real IBM processors are built on a "Heavy-Hex" lattice; a qubit is only physically connected to 2 or 3 neighbors. If our circuit requires a CNOT between two qubits on opposite sides of the chip, the transpiler must dynamically insert `SWAP` gates to route the quantum information across the lattice until the qubits are adjacent.

By using the modern `SamplerV2` and the `generate_preset_pass_manager`, we allow Qiskit to optimize our abstract mathematical algorithm into a physical blueprint that the specific IBM processor can understand.

In [4]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath('..'))

In [7]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from quantum_qr.dj import dj_circuit, oracle_from_secret_string

# 1. Build an authentic n=4 circuit (expected result: '0000')
# We use an all-zero oracle string to simulate an authentic QR code
dj = dj_circuit(oracle_from_secret_string("0000"), n=4)

# 2. Transpile for your specific hardware backend
# Optimization level 1 is usually sufficient for simple circuits like DJ
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa = pm.run(dj)

print(f"Original logical depth: {dj.depth()}")
print(f"Transpiled physical depth: {isa.depth()}")

# 3. Submit to the Sampler Primitive
sampler = Sampler(backend)
job = sampler.run([(isa,)], shots=1024)

print(f"✅ Job submitted!")
print(f"🆔 Job ID: {job.job_id()}")

Original logical depth: 5
Transpiled physical depth: 3
✅ Job submitted!
🆔 Job ID: d8ji80o32u0s73f7imfg


In [9]:
# Print the available data registers to see the correct name
print("Available data registers:", pub_result.data)

# You can access the data by its actual name found in the print output above.
# If it says something like "c", "classical", or "meas", use that!
# For example, if it prints: Available data registers: DataBin(c=BitArray(shape=(1024,), num_bits=4))
# Then you would use: counts = pub_result.data.c.get_counts()

Available data registers: DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=4>))


In [15]:
import json
import os
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# 1. Ensure the 'data' directory exists
os.makedirs("data", exist_ok=True)

# 2. Extract counts (using the register 'c' we confirmed)
counts = pub_result.data.c.get_counts()

# 3. Save to disk
with open("data/hardware_run_n4.json", "w") as f:
    json.dump(counts, f, indent=2)

# 4. Analyze results
total_shots = sum(counts.values())
p_zeros = counts.get("0000", 0) / total_shots

print(f"Total shots: {total_shots}")
print(f"P(zeros) observed: {p_zeros:.4f}")
print(f"Raw counts: {counts}")

# 5. Visualize
plot_histogram(counts, title="Physical QPU Results (n=4 Authentic)")
plt.savefig("data/hardware_histogram.png", dpi=300)
plt.show()

Total shots: 1024
P(zeros) observed: 0.9863
Raw counts: {'0000': 1010, '0001': 7, '0100': 1, '0010': 4, '1000': 2}


<Figure size 640x480 with 0 Axes>